In [1]:
import numpy as np
import pandas as pd
import duckdb
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
import sys
sys.path.append('../src')
from config import DB_PATH

con = duckdb.connect(DB_PATH)
df  = con.execute("SELECT * FROM features").df()
print(f"Loaded: {df.shape}") 

Loaded: (375, 44)


In [2]:
exclude   = {"region", "weather_station", "year", "yield_tonnes"}
leak_kw   = ("risk", "deviation", "label")
feat_cols = [c for c in df.columns
             if c not in exclude
             and not any(k in c for k in leak_kw)]
print(f"Weather features: {len(feat_cols)}") 

Weather features: 40


In [3]:
years    = sorted(df["year"].unique())
all_true = []
all_pred = []

for test_year in years:
    train = df[df["year"] != test_year].copy()
    test  = df[df["year"] == test_year].copy()

    # Region mean from training fold ONLY — no leakage
    region_mean = train.groupby("region")["yield_tonnes"].mean()
    train["anomaly"] = train["yield_tonnes"] - train["region"].map(region_mean)
    test["anomaly"]  = test["yield_tonnes"]  - test["region"].map(region_mean)

    # Inner loop: pick the single best feature using remaining training folds
    inner_years = sorted(train["year"].unique())
    best_feat   = None
    best_mae    = float("inf")

    for feat in feat_cols:
        inner_true = []
        inner_pred = []

        for inner_test_year in inner_years:
            inner_train = train[train["year"] != inner_test_year]
            inner_test  = train[train["year"] == inner_test_year]

            if len(inner_train) < 5:
                continue

            X_tr = inner_train[[feat]].values
            y_tr = inner_train["anomaly"].values
            X_te = inner_test[[feat]].values
            y_te = inner_test["anomaly"].values

            m = RidgeCV(alphas=[0.1, 1, 10, 100])
            m.fit(X_tr, y_tr)
            inner_true.extend(y_te)
            inner_pred.extend(m.predict(X_te))

        if len(inner_true) < 3:
            continue

        mae = mean_absolute_error(inner_true, inner_pred)
        if mae < best_mae:
            best_mae  = mae
            best_feat = feat

    # Outer fold: train on full training set using best feature
    X_train = train[[best_feat]].values
    y_train = train["anomaly"].values
    X_test  = test[[best_feat]].values
    y_test  = test["anomaly"].values

    final_model = RidgeCV(alphas=[0.1, 1, 10, 100])
    final_model.fit(X_train, y_train)

    pred_anomaly = final_model.predict(X_test)
    pred_yield   = pred_anomaly + test["region"].map(region_mean).values

    all_true.extend(test["yield_tonnes"].values)
    all_pred.extend(pred_yield)

    fold_mae = mean_absolute_error(test["yield_tonnes"].values, pred_yield)
    print(f"  {test_year}  best_feat={best_feat:<45}  MAE={fold_mae:.2f}") 

  2000  best_feat=planting_max_dry_streak                        MAE=8.96
  2001  best_feat=planting_max_dry_streak                        MAE=6.04
  2002  best_feat=planting_max_dry_streak                        MAE=4.45
  2003  best_feat=planting_max_dry_streak                        MAE=3.88
  2004  best_feat=planting_max_dry_streak                        MAE=3.35
  2005  best_feat=planting_max_dry_streak                        MAE=5.78
  2006  best_feat=planting_max_dry_streak                        MAE=8.41
  2007  best_feat=planting_max_dry_streak                        MAE=4.30
  2008  best_feat=planting_max_dry_streak                        MAE=9.67
  2009  best_feat=planting_max_dry_streak                        MAE=4.39
  2010  best_feat=planting_max_dry_streak                        MAE=5.44
  2011  best_feat=planting_max_dry_streak                        MAE=3.67
  2012  best_feat=planting_max_dry_streak                        MAE=3.52
  2013  best_feat=planting_max_dry_str

In [4]:
all_true = np.array(all_true)
all_pred = np.array(all_pred)

mae = mean_absolute_error(all_true, all_pred)
r2  = r2_score(all_true, all_pred)

print(f"\n{'='*50}")
print(f"NESTED-CV 1-FEATURE RIDGE — FINAL RESULTS")
print(f"{'='*50}")
print(f"  MAE : {mae:.3f} tonnes")
print(f"  R²  : {r2:.3f}")
print(f"{'='*50}")


NESTED-CV 1-FEATURE RIDGE — FINAL RESULTS
  MAE : 6.667 tonnes
  R²  : 0.069


In [5]:
# Re-run just to collect best features per fold
from collections import Counter

years    = sorted(df["year"].unique())
selected = []

for test_year in years:
    train = df[df["year"] != test_year].copy()
    region_mean = train.groupby("region")["yield_tonnes"].mean()
    train["anomaly"] = train["yield_tonnes"] - train["region"].map(region_mean)

    inner_years = sorted(train["year"].unique())
    best_feat   = None
    best_mae    = float("inf")

    for feat in feat_cols:
        inner_true, inner_pred = [], []
        for inner_test_year in inner_years:
            inner_train = train[train["year"] != inner_test_year]
            inner_test  = train[train["year"] == inner_test_year]
            if len(inner_train) < 5:
                continue
            m = RidgeCV(alphas=[0.1, 1, 10, 100])
            m.fit(inner_train[[feat]].values, inner_train["anomaly"].values)
            inner_true.extend(inner_test["anomaly"].values)
            inner_pred.extend(m.predict(inner_test[[feat]].values))
        if len(inner_true) < 3:
            continue
        mae = mean_absolute_error(inner_true, inner_pred)
        if mae < best_mae:
            best_mae  = mae
            best_feat = feat

    selected.append(best_feat)

print("Most selected features across all folds:")
for feat, count in Counter(selected).most_common(10):
    print(f"  {feat:<45} selected {count} times") 

Most selected features across all folds:
  planting_max_dry_streak                       selected 24 times
  growing_max_dry_streak                        selected 1 times


In [1]:
import duckdb
import sys
sys.path.append('../src')
from config import DB_PATH

con = duckdb.connect(DB_PATH)

tables = con.execute("SHOW TABLES").fetchall()
for (t,) in tables:
    con.execute(f"DROP TABLE IF EXISTS {t}")
    print(f"Dropped: {t}")

con.close()
print("\nAll tables dropped ✓") 

Dropped: clean_cotton
Dropped: clean_weather
Dropped: features
Dropped: features_with_risk
Dropped: predictions
Dropped: raw_cotton
Dropped: raw_weather

All tables dropped ✓


In [3]:
import numpy as np
import pandas as pd
import glob
import os
from collections import Counter
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import sys
sys.path.append('../src')
from config import RAW_WEATHER_DIR, RAW_COTTON_PATH, STAGES, COTTON_BASE_TEMP

# Load all available weather CSVs
csv_files = glob.glob(os.path.join(RAW_WEATHER_DIR, "*.csv"))
print(f"CSV files found: {len(csv_files)}")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

weather = pd.concat(dfs, ignore_index=True)
weather = weather[weather["year"].between(2000, 2024)]
print(f"Weather shape: {weather.shape}")
print(f"Districts: {weather['region'].nunique()}")
print(f"Columns: {list(weather.columns)}") 

CSV files found: 11
Weather shape: (100452, 12)
Districts: 11
Columns: ['date', 'temp_mean', 'temp_min', 'temp_max', 'precipitation', 'humidity_mean', 'wind_speed', 'et0', 'region', 'year', 'month', 'day']


In [4]:
cotton_raw = pd.read_excel(RAW_COTTON_PATH)
cotton_raw = cotton_raw.rename(columns={cotton_raw.columns[0]: "region"})
cotton_long = cotton_raw.melt(id_vars="region", var_name="year", value_name="yield_tonnes")
cotton_long["year"] = pd.to_numeric(cotton_long["year"], errors="coerce")
cotton_long = cotton_long.dropna(subset=["year"])
cotton_long["year"] = cotton_long["year"].astype(int)
cotton_long["yield_tonnes"] = pd.to_numeric(
    cotton_long["yield_tonnes"].replace(["-", "…", "..."], np.nan),
    errors="coerce"
)
cotton_long["region"] = cotton_long["region"].str.strip()

# Drop districts with any null
null_d = cotton_long[cotton_long["yield_tonnes"].isnull()]["region"].unique()
cotton_long = cotton_long[~cotton_long["region"].isin(null_d)]

# Keep only districts we have weather for
available = weather["region"].unique()
cotton = cotton_long[cotton_long["region"].isin(available)].copy()
cotton = cotton[cotton["year"].between(2000, 2024)].reset_index(drop=True)

print(f"Cotton shape: {cotton.shape}")
print(f"Districts matched: {cotton['region'].nunique()}")
print(f"Districts: {sorted(cotton['region'].unique())}")

Cotton shape: (275, 3)
Districts matched: 11
Districts: ['Aghdam district', 'Aghjabadi district', 'Beylagan district', 'Goranboy district', 'Imishli district', 'Kurdamir district', 'Saatli district', 'Sabirabad district', 'Tartar district', 'Yevlakh district', 'Zardab district']


C:\Users\User\AppData\Local\Temp\ipykernel_18684\2068634350.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cotton_long["yield_tonnes"].replace(["-", "…", "..."], np.nan),


In [5]:
weather["date"] = pd.to_datetime(weather["date"])
weather["month"] = weather["date"].dt.month

def gdd(row):
    return max(((row["temp_max"] + row["temp_min"]) / 2) - COTTON_BASE_TEMP, 0)

def dry_streak(s):
    best = cur = 0
    for v in s:
        cur = cur + 1 if v < 1 else 0
        best = max(best, cur)
    return best

def stage_features(stage_df, name):
    if stage_df.empty:
        return {}
    gdd_vals = stage_df.apply(gdd, axis=1)
    return {
        f"{name}_temp_mean":        round(stage_df["temp_mean"].mean(), 4),
        f"{name}_temp_min_mean":    round(stage_df["temp_min"].mean(), 4),
        f"{name}_temp_max_mean":    round(stage_df["temp_max"].mean(), 4),
        f"{name}_heat_stress_days": int((stage_df["temp_max"] > 35).sum()),
        f"{name}_frost_days":       int((stage_df["temp_min"] < 0).sum()),
        f"{name}_GDD":              round(gdd_vals.sum(), 4),
        f"{name}_total_rain":       round(stage_df["precipitation"].sum(), 4),
        f"{name}_rainy_days":       int((stage_df["precipitation"] > 1).sum()),
        f"{name}_dry_days":         int((stage_df["precipitation"] < 1).sum()),
        f"{name}_max_dry_streak":   dry_streak(stage_df["precipitation"]),
        f"{name}_humidity_mean":    round(stage_df["humidity_mean"].mean(), 4),
        f"{name}_wind_mean":        round(stage_df["wind_speed"].mean(), 4),
        f"{name}_et0_total":        round(stage_df["et0"].sum(), 4),
    }

rows = []
for _, row in cotton.iterrows():
    region = row["region"]
    year   = int(row["year"])
    daily  = weather[(weather["region"] == region) & (weather["year"] == year)]
    if daily.empty:
        continue
    feat = {"region": region, "year": year, "yield_tonnes": row["yield_tonnes"]}
    for sname, (sm, em) in STAGES.items():
        s_df = daily[(daily["month"] >= sm) & (daily["month"] <= em)]
        feat.update(stage_features(s_df, sname))
    rows.append(feat)

df = pd.DataFrame(rows).dropna().reset_index(drop=True)
print(f"Features shape: {df.shape}")
print(f"Districts: {df['region'].nunique()}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")

exclude  = {"region", "year", "yield_tonnes"}
feat_cols = [c for c in df.columns if c not in exclude]
print(f"Feature columns: {len(feat_cols)}") 

C:\Users\User\AppData\Local\Temp\ipykernel_18684\3166242793.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weather["date"] = pd.to_datetime(weather["date"])


Features shape: (275, 55)
Districts: 11
Years: 2000 – 2024
Feature columns: 52


In [6]:
years    = sorted(df["year"].unique())
all_true, all_pred, selected = [], [], []

for test_year in years:
    train = df[df["year"] != test_year].copy()
    test  = df[df["year"] == test_year].copy()

    region_mean = train.groupby("region")["yield_tonnes"].mean()
    train["anomaly"] = train["yield_tonnes"] - train["region"].map(region_mean)
    test["anomaly"]  = test["yield_tonnes"]  - test["region"].map(region_mean)

    inner_years = sorted(train["year"].unique())
    best_feat, best_mae = None, float("inf")

    for feat in feat_cols:
        inner_true, inner_pred = [], []
        for inner_year in inner_years:
            itr = train[train["year"] != inner_year]
            ite = train[train["year"] == inner_year]
            if len(itr) < 5 or ite.empty:
                continue
            m = RidgeCV(alphas=[0.1, 1, 10, 100])
            m.fit(itr[[feat]].values, itr["anomaly"].values)
            inner_true.extend(ite["anomaly"].values)
            inner_pred.extend(m.predict(ite[[feat]].values))
        if len(inner_true) < 3:
            continue
        mae = mean_absolute_error(inner_true, inner_pred)
        if mae < best_mae:
            best_mae, best_feat = mae, feat

    selected.append(best_feat)
    m = RidgeCV(alphas=[0.1, 1, 10, 100])
    m.fit(train[[best_feat]].values, train["anomaly"].values)
    pred_yield = m.predict(test[[best_feat]].values) + test["region"].map(region_mean).values
    all_true.extend(test["yield_tonnes"].values)
    all_pred.extend(pred_yield)

all_true = np.array(all_true)
all_pred = np.array(all_pred)
print(f"=== APPROACH 1: Teacher's Nested-CV 1-Feature ===")
print(f"MAE : {mean_absolute_error(all_true, all_pred):.3f}")
print(f"R²  : {r2_score(all_true, all_pred):.3f}")
print(f"\nMost selected features:")
for feat, count in Counter(selected).most_common(5):
    print(f"  {feat:<50} {count} times")

=== APPROACH 1: Teacher's Nested-CV 1-Feature ===
MAE : 5.875
R²  : 0.280

Most selected features:
  harvest_wind_mean                                  25 times


In [7]:
df_train = df[df["year"] <= 2021].copy()
df_test  = df[df["year"] >  2021].copy()

region_mean = df_train.groupby("region")["yield_tonnes"].mean()
df_train["anomaly"] = df_train["yield_tonnes"] - df_train["region"].map(region_mean)
df_test["anomaly"]  = df_test["yield_tonnes"]  - df_test["region"].map(region_mean)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(df_train[feat_cols].values, df_train["anomaly"].values)
importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)

print("Top 10 features by RF importance:")
print(importances.head(10).to_string())

print(f"\n{'Top-N':<8} {'Test MAE':>10} {'Test R²':>10}")
print("-" * 30)
for top_n in [1, 3, 5, 10, 15, 20]:
    top_feats = importances.head(top_n).index.tolist()
    m = Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=10.0))])
    m.fit(df_train[top_feats].values, df_train["anomaly"].values)
    pred_yield = (
        m.predict(df_test[top_feats].values)
        + df_test["region"].map(region_mean).values
    )
    mae = mean_absolute_error(df_test["yield_tonnes"].values, pred_yield)
    r2  = r2_score(df_test["yield_tonnes"].values, pred_yield)
    print(f"Top-{top_n:<3}  {mae:>10.3f} {r2:>10.3f}") 

Top 10 features by RF importance:
harvest_total_rain            0.224242
growing_humidity_mean         0.076696
harvest_wind_mean             0.056770
planting_temp_mean            0.037134
harvest_temp_min_mean         0.035932
planting_temp_max_mean        0.032132
boll_forming_humidity_mean    0.029622
boll_forming_temp_min_mean    0.028018
boll_forming_wind_mean        0.024689
harvest_max_dry_streak        0.024153

Top-N      Test MAE    Test R²
------------------------------
Top-1        12.555    -14.571
Top-3         9.727     -9.104
Top-5        12.418    -15.209
Top-10       11.112    -12.723
Top-15       11.568    -13.902
Top-20       10.648    -11.686


In [1]:
import numpy as np
import pandas as pd
import duckdb
from collections import Counter
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
import sys
sys.path.append('../src')
from config import DB_PATH

con = duckdb.connect(DB_PATH)
df  = con.execute("SELECT * FROM features").df()

exclude   = {"region", "weather_station", "year", "yield_tonnes"}
leak_kw   = ("risk", "deviation", "label")
feat_cols = [c for c in df.columns
             if c not in exclude
             and not any(k in c for k in leak_kw)]

print(f"Loaded: {df.shape}")
print(f"Features: {len(feat_cols)}")

Loaded: (375, 56)
Features: 52


In [2]:
# Only train/evaluate on years where model was stable (2000-2018)
df_stable = df[df["year"] <= 2018].copy()

years     = sorted(df_stable["year"].unique())
all_true  = []
all_pred  = []
selected  = []

print(f"Using years: {years[0]} – {years[-1]}  ({len(years)} folds)")
print(f"Rows: {len(df_stable)}")

for test_year in years:
    train = df_stable[df_stable["year"] != test_year].copy()
    test  = df_stable[df_stable["year"] == test_year].copy()

    region_mean = train.groupby("region")["yield_tonnes"].mean()
    train["anomaly"] = train["yield_tonnes"] - train["region"].map(region_mean)
    test["anomaly"]  = test["yield_tonnes"]  - test["region"].map(region_mean)

    inner_years = sorted(train["year"].unique())
    best_feat, best_mae = None, float("inf")

    for feat in feat_cols:
        inner_true, inner_pred = [], []
        for inner_year in inner_years:
            itr = train[train["year"] != inner_year]
            ite = train[train["year"] == inner_year]
            if len(itr) < 5 or ite.empty:
                continue
            m = RidgeCV(alphas=[0.1, 1, 10, 100])
            m.fit(itr[[feat]].values, itr["anomaly"].values)
            inner_true.extend(ite["anomaly"].values)
            inner_pred.extend(m.predict(ite[[feat]].values))
        if len(inner_true) < 3:
            continue
        mae = mean_absolute_error(inner_true, inner_pred)
        if mae < best_mae:
            best_mae, best_feat = mae, feat

    selected.append(best_feat)
    m = RidgeCV(alphas=[0.1, 1, 10, 100])
    m.fit(train[[best_feat]].values, train["anomaly"].values)
    pred_yield = m.predict(test[[best_feat]].values) + test["region"].map(region_mean).values
    all_true.extend(test["yield_tonnes"].values)
    all_pred.extend(pred_yield)

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\n=== NESTED-CV (2000–2018 only, skipping 2019–2021) ===")
print(f"MAE : {mean_absolute_error(all_true, all_pred):.3f}")
print(f"R²  : {r2_score(all_true, all_pred):.3f}")
print(f"\nMost selected features:")
for feat, count in Counter(selected).most_common(5):
    print(f"  {feat:<50} {count} times")

Using years: 2000 – 2018  (19 folds)
Rows: 285

=== NESTED-CV (2000–2018 only, skipping 2019–2021) ===
MAE : 4.253
R²  : 0.240

Most selected features:
  planting_temp_min_mean                             9 times
  harvest_heat_stress_days                           4 times
  planting_max_dry_streak                            2 times
  planting_frost_days                                1 times
  growing_temp_min_mean                              1 times


In [3]:
# Train final model on 2000-2018 with best feature
best_feat = Counter(selected).most_common(1)[0][0]
print(f"Best feature: {best_feat}")

df_train = df[df["year"] <= 2018].copy()
df_pred  = df[df["year"].isin([2022, 2023, 2024])].copy()

region_mean = df_train.groupby("region")["yield_tonnes"].mean()
df_train["anomaly"] = df_train["yield_tonnes"] - df_train["region"].map(region_mean)
df_pred["anomaly"]  = df_pred["yield_tonnes"]  - df_pred["region"].map(region_mean)

m = RidgeCV(alphas=[0.1, 1, 10, 100])
m.fit(df_train[[best_feat]].values, df_train["anomaly"].values)

pred_anomaly = m.predict(df_pred[[best_feat]].values)
pred_yield   = pred_anomaly + df_pred["region"].map(region_mean).values
true_yield   = df_pred["yield_tonnes"].values

print(f"\n=== TEST ON 2022–2024 (skipped 2019–2021) ===")
print(f"MAE : {mean_absolute_error(true_yield, pred_yield):.3f}")
print(f"R²  : {r2_score(true_yield, pred_yield):.3f}")

Best feature: planting_temp_min_mean

=== TEST ON 2022–2024 (skipped 2019–2021) ===
MAE : 15.131
R²  : -22.590


In [4]:
print("\n=== COMPARISON ===")
print(f"{'Approach':<45} {'R²':>8}")
print("-" * 55)
print(f"{'Full LOYO (2000–2021)':<45} {'0.074':>8}")
print(f"{'Stable LOYO (2000–2018 only)':<45} {r2_score(all_true, all_pred):>8.3f}")
print(f"{'Test on 2022–2024 (trained on 2000–2018)':<45} {r2_score(true_yield, pred_yield):>8.3f}")
print()
print("If stable LOYO R² >> full LOYO R²:")
print("  → confirms 2019–2021 are the problem years")
print("If test on 2022–2024 is positive:")
print("  → model recovers after skipping the disrupted years")


=== COMPARISON ===
Approach                                            R²
-------------------------------------------------------
Full LOYO (2000–2021)                            0.074
Stable LOYO (2000–2018 only)                     0.240
Test on 2022–2024 (trained on 2000–2018)       -22.590

If stable LOYO R² >> full LOYO R²:
  → confirms 2019–2021 are the problem years
If test on 2022–2024 is positive:
  → model recovers after skipping the disrupted years


In [5]:
con.close()

In [3]:
import pandas as pd
import os
import sys
sys.path.append('../src')
from config import RAW_WEATHER_DIR

import glob
for f in sorted(glob.glob(os.path.join(RAW_WEATHER_DIR, "*.csv"))):
    df = pd.read_csv(f, nrows=1)
    has_doy = "doy" in df.columns
    has_sunshine = "sunshine" in df.columns
    print(f"{os.path.basename(f):<45} doy={has_doy}  sunshine={has_sunshine}") 

aghdam_district_daily.csv                     doy=False  sunshine=True
aghjabadi_district_daily.csv                  doy=True  sunshine=True
barda_district_daily.csv                      doy=True  sunshine=True
beylagan_district_daily.csv                   doy=True  sunshine=True
bilasuvar_district_daily.csv                  doy=True  sunshine=True
goranboy_district_daily.csv                   doy=False  sunshine=True
imishli_district_daily.csv                    doy=True  sunshine=True
kurdamir_district_daily.csv                   doy=False  sunshine=True
neftchala_district_daily.csv                  doy=True  sunshine=True
saatli_district_daily.csv                     doy=True  sunshine=True
sabirabad_district_daily.csv                  doy=True  sunshine=True
salyan_district_daily.csv                     doy=True  sunshine=True
tartar_district_daily.csv                     doy=False  sunshine=True
yevlakh_district_daily.csv                    doy=False  sunshine=True
zardab_district